# 00 — LLM Config

**Module notebook — definitions only, no side effects beyond loading `.env`.**

Loads environment variables and exposes `get_llm()`, used by every downstream module that talks to the LLM (summarizer, extractor, rag_engine).

This notebook is meant to be loaded with `%run ./00_llm_config.ipynb` from `main.ipynb` — do not run it standalone as your only entry point.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()


In [ ]:
from langchain_openai import ChatOpenAI

# Confirmed: google/gemma-4-31b-it is a real model on OpenRouter (released
# April 2026). Fixed the casing here to lowercase 'b' ("31b", not "31B") to
# match OpenRouter's actual slug — the uppercase form risked a 404 depending
# on how strictly the API matches model-slug case.
OPENROUTER_MODEL = "google/gemma-4-31b-it:novita"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

TRANSCRIPT_SINGLE_CALL_CHAR_LIMIT = 1_000_000
def get_llm() -> ChatOpenAI:
    """Build a ChatOpenAI client pointed at OpenRouter."""
    api_key = os.getenv("OPENROUTER_API_KEY", "").strip()
    if api_key.lower().startswith("bearer "):
        api_key = api_key[7:].strip()
    if not api_key:
        raise RuntimeError(
            "OPENROUTER_API_KEY is not set. Add an OpenRouter API key "
            "to the environment before running the AI summary steps."
        )

    return ChatOpenAI(
        model=os.getenv("OPENROUTER_MODEL", OPENROUTER_MODEL),
        openai_api_key=api_key,
        openai_api_base=OPENROUTER_BASE_URL,
        default_headers={
            "HTTP-Referer": "https://github.com/openai/openai-python",
            "X-Title": "AI Video Assistant",
        },
        temperature=0.3,
    )


## Output language + code-switching

Shared by the summarizer, extractor, and RAG engine so every generated piece of text (summary, action items, chat answers, ...) follows the same language rule. Add new languages here in one place.

In [ ]:
def output_language_instruction(language: str) -> str:
    """
    System-prompt fragment that controls the LLM's output language.

    For Arabic, this deliberately asks the model to code-switch: keep
    technical/programming terms in English (as Arabic-speaking developers
    naturally do) while writing everything else in fluent Arabic.
    """
    language = language.lower()
    if language == "arabic":
        return (
            "IMPORTANT — output language: write your entire response in Arabic. "
            "However, keep technical, programming, or domain-specific terms "
            "(e.g. 'loop', 'variable', 'data type', tool/library names, jargon) "
            "in English exactly as an Arabic-speaking developer would naturally "
            "say them in conversation — do not translate these terms into Arabic. "
            "Everything else (sentence structure, explanations, connecting words) "
            "should be natural, fluent Arabic."
        )
    return "IMPORTANT — output language: write your entire response in English."


### Sanity check (safe to leave — prints no secrets)


In [ ]:
print(f"OpenRouter configuration loaded: {OPENROUTER_MODEL}")
print("OPENROUTER_API_KEY configured:", bool(os.getenv("OPENROUTER_API_KEY")))
